# 📓 Notebook 8 — Matplotlib Basics: Visualising Data

> **Module:** Data Science Libraries · **Estimated time:** 35–45 min · **Difficulty:** Beginner

A plot answers questions that tables cannot. Distributions, trends, outliers, relationships — they are *seen* before they are *understood*. Matplotlib is the foundational plotting library of Python: most other plotting libraries (seaborn, pandas\' built-in `.plot`, scikit-learn\'s visualisation helpers, …) are built on top of it.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. Use the **Figure / Axes** model — the right way to build matplotlib plots.
2. Choose the right chart type for the job: line, scatter, bar, hist, box, heatmap.
3. Add **titles, axis labels, legends, grids** and annotations.
4. Build **multi-panel** layouts with `subplots`.
5. Customise **colours, line styles, markers, alpha**.
6. **Save** publication-ready figures.
7. Recognise common matplotlib pitfalls and fix them.

## ✅ Prerequisites

Notebooks 1–7 (especially NumPy).

## 1. Setup — the canonical imports

Just three lines, and they are the same in every notebook you will ever see.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A consistent, professional look. Run once at the top of any notebook.
plt.rcParams.update({
    "figure.figsize"   : (8, 5),
    "figure.dpi"       : 100,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
})

print(f"matplotlib version: {plt.matplotlib.__version__}")

## 2. The Figure / Axes mental model

Matplotlib has two layers you need to understand:

```
┌────────────── Figure (the whole window) ──────────────┐
│                                                        │
│   ┌────── Axes (one plot inside) ──────┐               │
│   │                                    │               │
│   │   data lives here                  │               │
│   │                                    │               │
│   └────────────────────────────────────┘               │
│                                                        │
│   ┌────── Axes (another plot) ─────────┐               │
│   │                                    │               │
│   └────────────────────────────────────┘               │
└────────────────────────────────────────────────────────┘
```

- A **Figure** is the canvas. It may contain one or many sub-plots.
- An **Axes** is a single sub-plot — a coordinate system you draw on.

There are two coding styles in matplotlib:

| Style                       | Looks like                                  | When to use                  |
|-----------------------------|---------------------------------------------|------------------------------|
| State-based (`plt.plot`)    | `plt.plot(...); plt.title(...)`             | quick one-off plots          |
| Object-oriented (`ax.plot`) | `fig, ax = plt.subplots(); ax.plot(...)`    | **everything serious**       |

**Recommendation:** learn and use the object-oriented style from day one. It scales to multi-panel figures cleanly.

## 3. Your first line plot

In [ ]:
x = np.linspace(0, 2 * np.pi, 100)
y = np.sin(x)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y, color="#4C72B0", linewidth=2)
ax.set_title("First plot: sin(x)")
ax.set_xlabel("x")
ax.set_ylabel("sin(x)")
plt.tight_layout()
plt.show()

The seven lines above are essentially every plot you will ever make:

1. `fig, ax = plt.subplots(...)`  — create a Figure and one Axes.
2. `ax.plot(...)`  — draw the data.
3. `ax.set_title / set_xlabel / set_ylabel`  — label it.
4. `plt.tight_layout()`  — fix overlapping labels.
5. `plt.show()`  — render.

Everything else is variations on this recipe.

## 4. Multiple lines, legends, line styles

In [ ]:
x  = np.linspace(0, 2 * np.pi, 200)
y1 = np.sin(x)
y2 = np.cos(x)
y3 = np.sin(2 * x)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, y1, label="sin(x)",    color="#4C72B0", linewidth=2)
ax.plot(x, y2, label="cos(x)",    color="#DD8452", linewidth=2, linestyle="--")
ax.plot(x, y3, label="sin(2x)",   color="#55A467", linewidth=2, linestyle=":")
ax.axhline(0, color="grey", linewidth=0.7)

ax.set_title("Three trigonometric functions")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

> 💡 Common line styles: `"-"` solid, `"--"` dashed, `":"` dotted, `"-."` dash-dot. Markers: `"o"` circle, `"s"` square, `"^"` triangle, `"x"` cross, `"."` point.

For colour, use named CSS colours (`"crimson"`), hex codes (`"#4C72B0"`) or any standard matplotlib name. Stick to a small, consistent palette — your audience will thank you.

## 5. Scatter plots — for relationships between two variables

In [ ]:
rng = np.random.default_rng(42)
n = 80

# Synthetic relationship: y is roughly proportional to x plus noise
x = rng.uniform(0, 10, size=n)
y = 2.0 * x + rng.normal(0, 2.5, size=n)

# Add a third "feature" used for colour
quality = rng.integers(1, 6, size=n)

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(x, y, c=quality, cmap="viridis", s=60, edgecolor="black", alpha=0.8)

ax.set_title("Scatter: y ≈ 2x + noise (colour = quality 1–5)")
ax.set_xlabel("x")
ax.set_ylabel("y")

# Best-fit line (a one-line linear regression with numpy)
coef = np.polyfit(x, y, deg=1)
xs   = np.linspace(x.min(), x.max(), 50)
ax.plot(xs, coef[0] * xs + coef[1], color="red", linestyle="--",
        label=f"fit: y = {coef[0]:.2f}x + {coef[1]:.2f}")

fig.colorbar(sc, ax=ax, label="Quality rating")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Reading the chart.** The cloud of points slopes upward → x and y are positively correlated. The colour-encoded third variable shows whether there is any visible pattern in *quality* (here: not really). The red dashed line is the fitted regression. We will meet `np.polyfit` again in Notebook 9 inside scikit-learn\'s `LinearRegression`.

## 6. Bar charts — for comparing categories

In [ ]:
cities = ["Berlin", "Munich", "Hamburg", "Cologne", "Frankfurt"]
populations = [3.66, 1.49, 1.85, 1.09, 0.76]   # in millions

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(cities, populations,
              color=["#4C72B0", "#DD8452", "#55A467", "#C44E52", "#8172B2"],
              edgecolor="black")

# Annotate each bar with its value
for bar, val in zip(bars, populations):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f"{val:.2f}", ha="center", fontsize=10)

ax.set_title("Population of major German cities (millions)")
ax.set_ylabel("Population (millions)")
ax.set_ylim(0, max(populations) * 1.15)
plt.tight_layout()
plt.show()

### Horizontal bar charts

Use `barh` whenever the category labels are long — they read better horizontally.

In [ ]:
skills = ["Python", "SQL", "Statistics", "Machine Learning",
          "Communication", "Domain Knowledge", "Visualisation"]
importance = [95, 88, 92, 85, 90, 78, 82]

fig, ax = plt.subplots(figsize=(8, 5))
order = np.argsort(importance)
ax.barh(np.array(skills)[order], np.array(importance)[order],
        color="#4C72B0", edgecolor="black")

ax.set_title("Self-reported importance of data-science skills (0–100)")
ax.set_xlabel("Importance")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## 7. Histograms — for distributions

In [ ]:
rng = np.random.default_rng(0)

# Two simulated test-score populations
normal_class = rng.normal(loc=75, scale=10, size=400)
honors_class = rng.normal(loc=88, scale=5,  size=200)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(normal_class, bins=25, alpha=0.65, label="Normal class",
        color="#4C72B0", edgecolor="black")
ax.hist(honors_class, bins=25, alpha=0.65, label="Honors class",
        color="#DD8452", edgecolor="black")

# Vertical lines for means
ax.axvline(normal_class.mean(), color="#4C72B0", linestyle="--",
           label=f"Normal mean = {normal_class.mean():.1f}")
ax.axvline(honors_class.mean(), color="#DD8452", linestyle="--",
           label=f"Honors mean = {honors_class.mean():.1f}")

ax.set_title("Distribution of test scores by class")
ax.set_xlabel("Score")
ax.set_ylabel("Number of students")
ax.legend()
plt.tight_layout()
plt.show()

**Reading a histogram.** The x-axis is the value, the y-axis the count of observations in each bin. Two overlapping histograms with `alpha=0.65` is a clean way to compare two distributions. Vertical dashed lines highlight the means.

## 8. Box plots — distribution at a glance

In [ ]:
rng = np.random.default_rng(1)
classes = ["Math", "Science", "English", "History"]
scores  = [rng.normal(75, 12, 60),
           rng.normal(80,  8, 60),
           rng.normal(72, 15, 60),
           rng.normal(78, 10, 60)]

fig, ax = plt.subplots(figsize=(8, 5))
box = ax.boxplot(scores, tick_labels=classes, patch_artist=True,
                 medianprops=dict(color="black", linewidth=2))

colors = ["#4C72B0", "#55A467", "#DD8452", "#C44E52"]
for patch, c in zip(box["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)

ax.set_title("Score distribution by subject")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

**Reading a box plot.** The box covers the interquartile range (Q1 to Q3). The line inside is the **median**. Whiskers extend to data within 1.5×IQR; points beyond are outliers (drawn as dots).

## 9. Multi-panel layouts — subplots

In [ ]:
rng = np.random.default_rng(7)

# A small "dashboard" — 4 different views of the same data
data = rng.normal(loc=50, scale=15, size=300)
time  = np.arange(300)
trend = np.cumsum(rng.normal(0, 1, 300))

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("Mini exploratory dashboard", fontsize=14, fontweight="bold")

# (0, 0) Histogram
axes[0, 0].hist(data, bins=25, color="#4C72B0", edgecolor="black")
axes[0, 0].set_title("Distribution")
axes[0, 0].set_xlabel("Value")
axes[0, 0].set_ylabel("Count")

# (0, 1) Time series
axes[0, 1].plot(time, trend, color="#DD8452")
axes[0, 1].set_title("Random walk")
axes[0, 1].set_xlabel("Time")
axes[0, 1].set_ylabel("Cumulative value")

# (1, 0) Box plot
axes[1, 0].boxplot(data, vert=False, patch_artist=True,
                   boxprops=dict(facecolor="#55A467", alpha=0.6))
axes[1, 0].set_title("Box plot")
axes[1, 0].set_xlabel("Value")

# (1, 1) Scatter (data vs time)
axes[1, 1].scatter(time, data, alpha=0.5, color="#C44E52", s=10)
axes[1, 1].set_title("Value over time")
axes[1, 1].set_xlabel("Time")
axes[1, 1].set_ylabel("Value")

plt.tight_layout()
plt.show()

> 💡 `axes` is a NumPy array of Axes objects — `axes[0, 0]` is top-left, `axes[1, 1]` is bottom-right. For a single row or column you get a 1-D array (`axes[0]`, `axes[1]`).

## 10. Annotations — making the punchline visible

In [ ]:
years  = np.arange(2014, 2026)
values = np.array([50, 52, 60, 65, 68, 75, 90, 82, 95, 115, 130, 145])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(years, values, marker="o", color="#4C72B0", linewidth=2)

# Highlight the COVID dip
ax.annotate("COVID dip\n(2022)",
            xy=(2022, 82), xytext=(2020, 50),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10, ha="center")

# Highlight the most recent jump
ax.annotate("Strong growth\n2024-2025",
            xy=(2025, 145), xytext=(2022, 130),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10, ha="center")

ax.set_title("Annual revenue (illustrative)")
ax.set_xlabel("Year")
ax.set_ylabel("Revenue (€M)")
plt.tight_layout()
plt.show()

## 11. A heatmap — for matrices

In [ ]:
# Correlation matrix of 5 random features (synthetic)
rng = np.random.default_rng(0)
features = ["f1", "f2", "f3", "f4", "f5"]
X = rng.normal(size=(200, 5))
X[:, 1] = 0.8 * X[:, 0] + 0.2 * X[:, 1]    # f2 correlated with f1
X[:, 3] = -0.5 * X[:, 2] + 0.5 * X[:, 3]   # f4 anti-correlated with f3

corr = np.corrcoef(X.T)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(features)), features)
ax.set_yticks(range(len(features)), features)
ax.set_title("Feature correlation matrix")

# Annotate each cell
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr[i, j]) > 0.6 else "black", fontsize=9)

fig.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

## 12. Saving figures

Drop `plt.savefig("name.png", dpi=200, bbox_inches="tight")` *before* `plt.show()` to save your figure to disk. Common formats:

| Extension | When to use                                          |
|-----------|------------------------------------------------------|
| `.png`    | reports, web, default raster                          |
| `.pdf`    | papers, slides, infinitely scalable vectors           |
| `.svg`    | web vectors / further editing in Inkscape, Illustrator |

`bbox_inches="tight"` crops out unused whitespace; `dpi=200` (or 300) gives a sharp result.

## 13. Common pitfalls

| Pitfall                                       | Symptom                                | Fix |
|-----------------------------------------------|----------------------------------------|-----|
| Figures look squished / overlap labels        | crowded titles / labels                | call `plt.tight_layout()` before `plt.show()` |
| Calling `plt.plot` multiple times in same cell | unintended cumulative figure           | use `fig, ax = plt.subplots()` for each plot |
| Wrong axis order on heatmap                   | matrix looks rotated/mirrored          | remember `imshow` plots `data[row][col]`, with row 0 on top |
| Colour bars on subplots                       | colour bar covers a neighbour          | pass `ax=` to `fig.colorbar(im, ax=ax)` |
| Bad colour choices                            | unreadable / non-colourblind-friendly  | prefer perceptual maps like `"viridis"` or `"cividis"` |

## 🧪 Practice exercises

### Exercise 1 — Three lines, one plot

On the interval `x ∈ [0, 5]`, plot `y₁ = x`, `y₂ = x²`, `y₃ = 2ˣ` on the same axes. Add labels, a legend, a title, and a grid.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
x = np.linspace(0, 5, 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, x,        label="y = x",   linewidth=2)
ax.plot(x, x**2,     label="y = x²",  linewidth=2)
ax.plot(x, 2**x,     label="y = 2ˣ",  linewidth=2)

ax.set_title("Linear vs quadratic vs exponential growth")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()
```

**Interpretation.** The exponential `2ˣ` quickly outpaces the quadratic, which in turn quickly outpaces the linear. This is exactly why algorithm complexities like O(n²) and O(2ⁿ) feel so different in practice.
</details>

### Exercise 2 — Histogram with mean line

Generate 1 000 samples from a normal distribution with mean 100 and std 15. Plot a histogram, overlay vertical dashed lines for the mean and (mean ± std).

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(0)
data = rng.normal(100, 15, 1000)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(data, bins=30, color="#4C72B0", edgecolor="black", alpha=0.8)

m = data.mean()
s = data.std()
ax.axvline(m,      color="red",    linestyle="--", label=f"mean = {m:.1f}")
ax.axvline(m - s,  color="orange", linestyle=":")
ax.axvline(m + s,  color="orange", linestyle=":", label="±1 std")

ax.set_title("Normal(100, 15²) — 1 000 samples")
ax.set_xlabel("Value")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

### Exercise 3 — Subplots side-by-side

Create a figure with **two side-by-side subplots**:

1. Left: a bar chart of sales by region (sample data below).
2. Right: a pie chart of the same data.

Use the same colour palette for both, and add `fig.suptitle(...)`.

In [ ]:
# Your code here  👇
regions = ["North", "South", "East", "West"]
sales   = [1200, 950, 1450, 880]


<details>
<summary>💡 <b>Solution</b></summary>

```python
palette = ["#4C72B0", "#DD8452", "#55A467", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle("Sales by region", fontsize=14, fontweight="bold")

# Bar
axes[0].bar(regions, sales, color=palette, edgecolor="black")
axes[0].set_title("Bar")
axes[0].set_ylabel("Sales (€)")

# Pie
axes[1].pie(sales, labels=regions, colors=palette, autopct="%.1f%%",
            wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("Share")

plt.tight_layout()
plt.show()
```

**Note.** Pie charts are great for *parts of a whole* with 3–6 categories — beyond that, prefer a horizontal bar chart.
</details>

### Exercise 4 — Annotated scatter

Generate 100 random points (x, y) with `y = 0.5*x + noise`. Plot them, add the regression line, and annotate the **largest residual** (the point furthest from the line).

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(3)
x = rng.uniform(0, 10, 100)
y = 0.5 * x + rng.normal(0, 1.0, 100)

# Fit
m, b = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 50)
y_pred = m * x + b
residuals = y - y_pred

# Worst point
i = np.argmax(np.abs(residuals))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.6)
ax.plot(xs, m * xs + b, "r--", label=f"y = {m:.2f}x + {b:.2f}")
ax.scatter([x[i]], [y[i]], color="black", s=80, zorder=5, label="largest residual")
ax.annotate(f"residual = {residuals[i]:.2f}", xy=(x[i], y[i]),
            xytext=(x[i] + 0.3, y[i] + 0.5),
            arrowprops=dict(arrowstyle="->"))

ax.set_title("Linear fit and the worst-fit point")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

### Exercise 5 — Debug me 🐞

The plot below should show *two* sine curves with different phases. It currently only shows one and has no axis labels. Fix it.

In [ ]:
x = np.linspace(0, 2*np.pi, 200)

plt.figure()
plt.plot(x, np.sin(x))
plt.plot(x, np.sin(x))
plt.title("Two sine curves")
plt.show()


<details>
<summary>💡 <b>Solution</b></summary>

Two issues: both lines plot the same `sin(x)`, and the axes are unlabelled.

```python
x = np.linspace(0, 2*np.pi, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, np.sin(x),         label="sin(x)")
ax.plot(x, np.sin(x + np.pi/4), label="sin(x + π/4)")
ax.set_title("Two sine curves with a phase shift")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

## 🎁 Bonus mini-project — Build a dashboard

Generate a synthetic dataset of 500 customers with: `age`, `income`, `spending`, `region`. Then build a 2×2 dashboard:

1. Histogram of `age`.
2. Scatter `income` vs `spending`, coloured by `age`.
3. Bar chart of average `spending` per `region`.
4. Box plot of `income` per `region`.

In [ ]:
# Your code here  👇
rng = np.random.default_rng(0)
n = 500
age      = rng.integers(18, 75, n)
income   = rng.normal(loc=age * 800, scale=8_000, size=n).clip(min=1_000)
spending = 0.3 * income + rng.normal(0, 2_000, size=n)
region   = rng.choice(["North", "South", "East", "West"], size=n)


<details>
<summary>💡 <b>Solution</b></summary>

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Customer dashboard", fontsize=15, fontweight="bold")

# (0, 0) Age histogram
axes[0, 0].hist(age, bins=20, color="#4C72B0", edgecolor="black")
axes[0, 0].set_title("Age distribution")
axes[0, 0].set_xlabel("Age")
axes[0, 0].set_ylabel("Count")

# (0, 1) Income vs spending, coloured by age
sc = axes[0, 1].scatter(income, spending, c=age, cmap="viridis", alpha=0.7)
axes[0, 1].set_title("Income vs spending")
axes[0, 1].set_xlabel("Income (€)")
axes[0, 1].set_ylabel("Spending (€)")
fig.colorbar(sc, ax=axes[0, 1], label="Age")

# (1, 0) Mean spending per region
regions = sorted(set(region))
means = [spending[region == r].mean() for r in regions]
axes[1, 0].bar(regions, means, color=["#4C72B0","#DD8452","#55A467","#C44E52"], edgecolor="black")
axes[1, 0].set_title("Mean spending per region")
axes[1, 0].set_ylabel("€")

# (1, 1) Income per region — boxplot
inc_per_region = [income[region == r] for r in regions]
axes[1, 1].boxplot(inc_per_region, tick_labels=regions, patch_artist=True)
axes[1, 1].set_title("Income distribution per region")
axes[1, 1].set_ylabel("€")

plt.tight_layout()
plt.show()
```
</details>

## 🧠 Key takeaways

1. Use the **Figure / Axes** (`fig, ax = plt.subplots()`) style for everything beyond a quick one-off.
2. Always label your axes and give the plot a title — a chart without context is a riddle.
3. Match the **chart type** to the question:
   - line for *trend over time*,
   - bar for *comparing categories*,
   - histogram / box for *distributions*,
   - scatter for *relationships*,
   - heatmap for *matrices*.
4. Build multi-panel layouts with `plt.subplots(rows, cols)` for "data dashboards".
5. Annotate the punchline — arrows, vertical lines, callouts.
6. Save with `plt.savefig("name.png", dpi=200, bbox_inches="tight")`.
7. Stick to a small, perceptually uniform colour palette (`viridis`, `cividis`).

## ✅ Self-assessment

- [ ] Build a labelled line plot with multiple lines and a legend.
- [ ] Choose between a bar chart, hist, scatter, and box plot.
- [ ] Create a 2×2 subplot layout.
- [ ] Add an annotation with an arrow.
- [ ] Save a figure to PNG.

## 🚀 Next step

Continue with **Notebook 9 — Scikit-Learn Basics** to build your first machine-learning models.